# HFO Evaluation on Real-Time sEEG Data using an snn-torch-trained Network (Fixed-Precision)
This notebook shows how to evaluate the performance of an SNN loaded into loihi with fixed-point precision on real-time sEEG data. This will still run on a CPU simulation, but the precision will be fixed to emulate the behavior of a Loihi chip, which is implemented on a separate GitHub Repo.

## Check WD (change if necessary) and file loading

In [1]:
# Show current directory
import os
curr_dir = os.getcwd()
print(curr_dir)

# Check if the current WD is the file location
if "/thesis-lava/src/nir" not in os.getcwd():
    # Set working directory to this file location
    file_location = f"{os.getcwd()}/thesis-lava/src/nir"
    print("File Location: ", file_location)

    # Change the current working Directory
    os.chdir(file_location)

    # New Working Directory
    print("New Working Directory: ", os.getcwd())

/home/monkin/Desktop/feup/thesis
File Location:  /home/monkin/Desktop/feup/thesis/thesis-lava/src/nir
New Working Directory:  /home/monkin/Desktop/feup/thesis/thesis-lava/src/nir


### Add Parent Directory to Path
Need to add it to access `utils` without adding it to PATH

In [2]:
# Add grandparent directory to path (To acess sntt_utils)
import sys

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
# Add the grandparent directory to the system path
# grandparent_dir = os.path.abspath(os.path.join(current_dir, os.pardir, os.pardir))
# sys.path.append(parent_dir)
sys.path.insert(0, parent_dir)


print(sys.path)

['/home/monkin/Desktop/feup/thesis/thesis-lava/src', '/home/monkin/Desktop/feup/thesis', '/home/monkin/Desktop/feup/ncn-lava/src', '/home/monkin/Desktop/feup/thesis/thesis-lava/src', '/home/monkin/Desktop/feup/thesis/lava-dl/src', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/home/monkin/Desktop/feup/thesis/.venv/lib/python3.10/site-packages', '/home/monkin/Desktop/feup/thesis/lava-dl', '/home/monkin/Desktop/feup/thesis/.venv/src/lava/src', '/home/monkin/Desktop/feup/thesis/.venv/src/lava']


## Check if Cuda is available

In [3]:
import torch
import numpy as np

# Check CUDA Installation
print(torch.cuda.is_available())

# Get the number of available GPUs
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs: {num_gpus}")

# Get information about each GPU
for i in range(num_gpus):
    device_props = torch.cuda.get_device_properties(i)
    print(f"\nGPU {i}:")
    print(f"  Name: {device_props.name}")
    print(f"  Total memory: {device_props.total_memory / 1024**3:.2f} GB")
    print(f"  Multiprocessor count: {device_props.multi_processor_count}")
    print(f"  Major compute capability: {device_props.major}")
    print(f"  Minor compute capability: {device_props.minor}")

True
Number of GPUs: 1

GPU 0:
  Name: NVIDIA GeForce RTX 3060 Ti
  Total memory: 7.77 GB
  Multiprocessor count: 38
  Major compute capability: 8
  Minor compute capability: 6


### Define the Device that will be used to train the SNN

In [4]:
# Set the device to be used
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")     # torch.device("cpu") #

print("device: ", device)

device:  cuda


## Load the Network from the `NIR` file

In [5]:
from utils.hfo import band_to_file_name, MarkerType, BaselineAlgorithm

# ----- Frequency Band Parameters -----
# Declare if using ripples, fast ripples, or both
chosen_band = MarkerType.RIPPLE     # RIPPLE, FAST_RIPPLE, or BOTH
# Specify the chosen Baseline Algorithm
chosen_baseline_alg_suffix = BaselineAlgorithm.MEDIAN # BaselineAlgorithm.EIGHTY_PERC    # BaselineAlgorithm.Q3
# Get the suffix for the chosen frequency band
BAND_FILENAME = band_to_file_name(chosen_band)

In [6]:
import numpy as np
import nir
import matplotlib.pyplot as plt

# Load the Trained Network from the NIR File
NIR_FILENAME = f"{BAND_FILENAME}_{chosen_baseline_alg_suffix}_trained_net.nir"

nir_network = nir.read(f"models/{NIR_FILENAME}")

### Analyze individual Layers of the imported Network

In [7]:
nirLIF1 = nir_network.nodes["lif1"]
nirLIF2 = nir_network.nodes["lif2"]
nirLIFOut = nir_network.nodes["lif_out"]
print(nirLIF1)

CubaLIF(tau_syn=array([0.00017387, 0.00024335, 0.00035938, 0.00018639, 0.00012913,
       0.00034115, 0.00020152, 0.00019603, 0.00024451, 0.00016147,
       0.00027003, 0.00019848, 0.00039838, 0.00038222, 0.00017285,
       0.00042039, 0.00034222, 0.00016927, 0.00025303, 0.00073434,
       0.00026798, 0.00015084, 0.00015396, 0.00026093], dtype=float32), tau_mem=array([0.0001626 , 0.00033893, 0.00011517, 0.00027778, 0.00022234,
       0.00035438, 0.00015415, 0.00022128, 0.00040636, 0.00038729,
       0.00012157, 0.00017172, 0.00017749, 0.00015703, 0.0002823 ,
       0.00053919, 0.00018323, 0.00039194, 0.00014772, 0.00018904,
       0.00019635, 0.00027382, 0.00015956, 0.00020556], dtype=float32), r=array([1.6259756, 3.3892968, 1.1517061, 2.7777927, 2.2234085, 3.5437503,
       1.5414665, 2.2127855, 4.0635676, 3.8729072, 1.2157393, 1.7172083,
       1.7748575, 1.5703335, 2.8229654, 5.391886 , 1.832302 , 3.919392 ,
       1.4771954, 1.8904446, 1.9635096, 2.7382052, 1.5955733, 2.0555692],
 

In [8]:
# Calculate the dv and du parameters from tau_mem and tau_syn
lif1_dv = np.array(list(map(lambda x: (1e-4 / x), nirLIF1.tau_mem)))
lif1_du = np.array(list(map(lambda x: (1e-4 / x), nirLIF1.tau_syn)))

lif2_dv = np.array(list(map(lambda x: (1e-4 / x), nirLIF2.tau_mem)))
lif2_du = np.array(list(map(lambda x: (1e-4 / x), nirLIF2.tau_syn)))

lif_out_dv = (1e-4 / nirLIFOut.tau_mem)
lif_out_du = (1e-4 / nirLIFOut.tau_syn)

In [9]:
nirLIFOut

CubaLIF(tau_syn=np.float32(0.00030641834), tau_mem=np.float32(0.00030641872), r=np.float32(3.0641873), v_leak=np.float32(0.0), v_threshold=np.float32(1.0), w_in=np.float32(3.0641835), input_type={'input': array([], dtype=float64)}, output_type={'output': array([], dtype=float64)}, metadata={}, num_neurons=np.int64(1))

In [10]:
# Print the network summary
print(f"NIR Network Info: Nº Nodes: {len(nir_network.nodes)} | Nº Edges: {len(nir_network.edges)}")

print("Nodes: ", nir_network.nodes)
print(f"\nLIF Nodes dv and du: LIF1: dv: {lif1_dv} | du: {lif1_du}\nLIF2: dv: {lif2_dv} | du: {lif2_du}\nLIFOut: dv: {lif_out_dv} | du: {lif_out_du}")

print("\nEdges: ", nir_network.edges)

NIR Network Info: Nº Nodes: 8 | Nº Edges: 7
Nodes:  {'fc3': Linear(weight=array([[-0.12796907,  0.6285288 , -0.12428566,  0.681862  ,  0.4625867 ,
        -0.01554998,  0.84482646,  0.68833286,  0.5512423 ,  0.6665629 ,
         0.80314606,  0.5501224 ,  0.66260606,  0.43678117,  0.73843795,
         0.56483155, -0.23407368, -0.16438858,  0.7274083 ,  0.4603684 ,
        -0.02943515,  0.50974905,  0.55587983, -0.25474668],
       [-0.15086836,  0.5077672 ,  0.62886876, -0.05198042,  0.70652866,
         0.44173464,  0.67742395, -0.18004082, -0.00871898,  0.5277912 ,
        -0.14156589,  0.40357798, -0.2910459 , -0.08679418,  0.44882917,
        -0.130766  , -0.01235545,  0.66155946,  0.50826025, -0.36246124,
        -0.11799964,  0.69233155, -0.03061154, -0.22221653],
       [ 0.5198871 ,  0.4037349 ,  0.6156272 , -0.2484523 ,  0.55329895,
        -0.02090882,  0.49627668, -0.409064  ,  0.51931995, -0.05392839,
        -0.29671165,  0.5371652 ,  0.7513609 ,  0.7248579 , -0.10710276,
 

In [11]:
from typing import TypedDict

class LifNode(TypedDict):
    tau_syn: np.float32
    tau_mem: np.float32
    r: np.float32
    v_leak: np.float32
    v_threshold: np.float32
    w_in: np.float32
    input_type: dict[str, list[np.float64]]
    output_type: dict[str, list[np.float64]]
    metadata: dict
    num_neurons: np.int64

nirLifOut: LifNode = nir_network.nodes["lif_out"]

## Transform the NIR Representation to a Lava Network

In [12]:
from nir_to_lava_edit import ImportConfig, LavaLibrary, import_from_nir
# from nir_to_lava import ImportConfig, LavaLibrary, import_from_nir

nir_dt = 1e-4
config = ImportConfig(
    dt=nir_dt, fixed_pt=False, on_chip=False, library_preference=LavaLibrary.Lava,
)

## Read the Input Data and the Ground Truth
For evaluation, we will only use the last 20% of each SEEG recording, corresponding to the Training Set.

In [13]:
IS_CLINICAL = False
PATIENT_LABEL = "csl"
INPUT_TYPE = "clinical" if IS_CLINICAL else "synthetic"

In [14]:
from utils.io import preview_np_array

# Define the channel suffix (Channel range indicates a certain Brain Region & SNR level)
CH_SUFFIX = "ch90-119"  # "ch90-119"

# Get the suffix for the chosen frequency band
BAND_FILENAME = band_to_file_name(chosen_band)

# Define the INPUT Folder
INPUT_FOLDER = f"input_data/{BAND_FILENAME}_{chosen_baseline_alg_suffix}_{CH_SUFFIX}"

In [15]:
# Load the input data
up_spikes = np.load(f"{INPUT_FOLDER}/up_spike_train.npy")
down_spikes = np.load(f"{INPUT_FOLDER}/down_spike_train.npy")
# Load the Ground Truth
gt_data = np.load(f"{INPUT_FOLDER}/gt_data.npy")

# Print the shape of the loaded data
print("Up Spikes Shape: ", up_spikes.shape)
print("Down Spikes Shape: ", down_spikes.shape)
print("GT Data Shape: ", gt_data.shape)

Up Spikes Shape:  (85918,)
Down Spikes Shape:  (86007,)
GT Data Shape:  (720,)


In [16]:
# Define np.array containing the timestep of GT events
gt_times = np.array([int(round(gt_elem[1])) for gt_elem in gt_data])

# Preview the data
preview_np_array(gt_times, "gt_times", edge_items=2)

gt_times Shape: (720,).
Preview: [   6967    9794 ... 3592313 3595019]


In [17]:
# See the time of the first and last UP / DN spikes
first_up, last_up = up_spikes[0], up_spikes[-1]
first_dn, last_dn = down_spikes[0], down_spikes[-1]

print(f"First Up Spike: {first_up} | Last Up Spike: {last_up}")
print(f"First Down Spike: {first_dn} | Last Down Spike: {last_dn}")

First Up Spike: 66.40625 | Last Up Spike: 3599981.93359375
First Down Spike: 63.4765625 | Last Down Spike: 3600000.0


## Define Important Parameters for the Evaluation

In [18]:
from utils.input import INPUT_DURATION, NUM_CH_PER_SNR 

# Simulation Time Parameters
num_steps_per_ch = int(INPUT_DURATION)     # int(INPUT_DURATION * NUM_CH_PER_SNR)    # Number of steps to run the simulation   # TODO: CAREFUL WITH THIS DURATION (INPUT_DURATION IS THE DURATION OF THE SYNTHETIC DATA)
num_channels = NUM_CH_PER_SNR
init_offset = 0 # 900 # 33400      #   
virtual_time_step_interval = 1  # dt = 1 ms

print(f"Num Steps: {num_steps_per_ch}")

Num Steps: 120000


### Split the Input Data and Ground Truth by the number of channels.

In [19]:
'''
Split the UP and DOWN Spikes by the number of channels.
Each Channel has NUM_STEPS_PER_CH timesteps
'''
up_spikes_per_ch, down_spikes_per_ch = np.ndarray(shape=(NUM_CH_PER_SNR,), dtype=object), np.ndarray(shape=(NUM_CH_PER_SNR,), dtype=object)
gt_times_per_ch = np.ndarray(shape=(NUM_CH_PER_SNR,), dtype=object)

for up_spike in up_spikes:
    ch_idx = int(up_spike // INPUT_DURATION)
    if up_spikes_per_ch[ch_idx] is None:
        # Remove the first None element
        up_spikes_per_ch[ch_idx] = np.array([up_spike - (ch_idx * INPUT_DURATION)], dtype=np.float64)
    else:
        up_spikes_per_ch[ch_idx] = np.append(up_spikes_per_ch[ch_idx], up_spike - (ch_idx * INPUT_DURATION))
for down_spike in down_spikes:
    ch_idx = int(down_spike // INPUT_DURATION)

    if ch_idx == 30:
        print(f"Edge Case: switching channel index from 30 to 29")
        ch_idx = 29
        
    if down_spikes_per_ch[ch_idx] is None:
        # Remove the first None element
        down_spikes_per_ch[ch_idx] = np.array([down_spike - (ch_idx * INPUT_DURATION)], dtype=np.float64)
    else:
        down_spikes_per_ch[ch_idx] = np.append(down_spikes_per_ch[ch_idx], down_spike - (ch_idx * INPUT_DURATION))
for gt_time in gt_times:
    ch_idx = int(gt_time // INPUT_DURATION)
    if gt_times_per_ch[ch_idx] is None:
        # Remove the first None element
        gt_times_per_ch[ch_idx] = np.array([gt_time - (ch_idx * INPUT_DURATION)], dtype=np.float64)
    else:
        gt_times_per_ch[ch_idx] = np.append(gt_times_per_ch[ch_idx], gt_time - (ch_idx * INPUT_DURATION))

# Print the shape of the loaded data
print("Up Spikes Per Channel Shape: ", up_spikes_per_ch.shape)
print("Down Spikes Per Channel Shape: ", down_spikes_per_ch.shape)
print("GT Times Per Channel Shape: ", gt_times_per_ch.shape)

Edge Case: switching channel index from 30 to 29
Up Spikes Per Channel Shape:  (30,)
Down Spikes Per Channel Shape:  (30,)
GT Times Per Channel Shape:  (30,)


### Check if the Input data and Ground Truth is well-distributed among the channels

In [20]:
# Check if UP spikes are more or less distributed by the INPUT CHANNELS
for ch_idx in range(NUM_CH_PER_SNR):
    ch_up_spikes = up_spikes_per_ch[ch_idx].size
    ch_down_spikes = down_spikes_per_ch[ch_idx].size
    ch_gt_times = gt_times_per_ch[ch_idx].size
    print(f"""Nº UP / DN Spikes for Channel {ch_idx}: {ch_up_spikes}({round((ch_up_spikes / up_spikes.size)*100, 2)}%) / {ch_down_spikes}({round((ch_down_spikes / down_spikes.size)*100, 2)}%)
          | Nº GT Times: {ch_gt_times}({round((ch_gt_times / gt_times.size)*100, 2)}%)""")

Nº UP / DN Spikes for Channel 0: 2923(3.4%) / 2929(3.41%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 1: 2971(3.46%) / 2972(3.46%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 2: 2617(3.05%) / 2628(3.06%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 3: 2912(3.39%) / 2907(3.38%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 4: 2883(3.36%) / 2899(3.37%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 5: 2611(3.04%) / 2620(3.05%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 6: 3025(3.52%) / 3030(3.52%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 7: 3175(3.7%) / 3189(3.71%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 8: 2715(3.16%) / 2710(3.15%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 9: 2991(3.48%) / 2986(3.47%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 10: 2878(3.35%) / 2875(3.34%)
  

As we can see, the spikes were well distributed.

Now, we can perform the remaining steps inside a loop to detect the HFOs 1 sEEG Channel at a time to reduce the memory usage.

## Get the Input Data and Ground Truth of the Testing Data Set

In [21]:
from utils.snn import train_test_split_seeg_data, TRAIN_SPLIT
from math import floor

train_ind_input_duration = floor(INPUT_DURATION * TRAIN_SPLIT)  # 80% of the input duration for training
test_ind_input_duration = INPUT_DURATION - train_ind_input_duration # 20% of the input duration for testing
print(f"Train Input Duration: {train_ind_input_duration} | Test Input Duration: {test_ind_input_duration}")

Train Input Duration: 96000 | Test Input Duration: 24000


In [22]:
# Split the data into training and testing sets
# Shape of the data after splitting: (num_channels, num_events_per_channel)
_, test_up_spike_evts, _, test_down_spike_evts, _, test_gt_times = train_test_split_seeg_data(
            up_spikes_per_ch, down_spikes_per_ch, gt_times_per_ch,
            single_ch_duration=INPUT_DURATION,
            train_single_ch_duration=train_ind_input_duration,
            test_single_ch_duration=test_ind_input_duration
)

# Print the shape of the loaded data
preview_np_array(test_up_spike_evts, "Test Up Spike Events", edge_items=2)
preview_np_array(test_down_spike_evts, "Test Down Spike Events", edge_items=2)
preview_np_array(test_gt_times, "Test GT Times", edge_items=2)

Test Up Spike Events Shape: (30,).
Preview: [array([   41.015625  ,    89.84375   , ..., 23860.83984375,
        23909.66796875], shape=(644,))
 array([  291.015625  ,   300.78125   , ..., 23932.6171875 ,
        23976.07421875], shape=(628,))                       ...
 array([   60.05859375,    68.84765625, ..., 23949.70703125,
        23967.28515625], shape=(653,))
 array([4.88281250e+00, 1.61132812e+01, ..., 2.39575195e+04,
        2.39819336e+04], shape=(585,))                      ]
Test Down Spike Events Shape: (30,).
Preview: [array([   37.59765625,    85.9375    , ..., 23858.3984375 ,
        23907.71484375], shape=(652,))
 array([  293.45703125,   304.6875    , ..., 23935.05859375,
        23979.4921875 ], shape=(633,))                       ...
 array([   62.98828125,    74.70703125, ..., 23945.3125    ,
        23954.1015625 ], shape=(654,))
 array([8.78906250e+00, 2.24609375e+01, ..., 2.39545898e+04,
        2.39628906e+04], shape=(586,))                      ]
Test GT Time

# HFO Evaluation

In [23]:
from detect_hfo import evaluate_hfo_detector, HFOEvalResults

# Store the results of the HFO Evaluation for each sEEG Channel
prediction_results: list[HFOEvalResults] = []
for ch_idx in range(NUM_CH_PER_SNR):
    # Get the Input Data for the current channel
    curr_spk_evts = np.array(
        [test_up_spike_evts[ch_idx], test_down_spike_evts[ch_idx]], dtype=object)
    
    # Get the GT Times for the current channel
    curr_gt_times = test_gt_times[ch_idx]
    
    EVAL_LABEL = f"CH Index: {ch_idx}"

    USE_REFRAC = True  # Whether to use Refractory Neurons in the Output Layer

    # Run the HFO Evaluation
    print(f"\n\n--- HFO Evaluation - {EVAL_LABEL} ---")
    curr_eval_results = evaluate_hfo_detector(
        nir_network, config, chosen_band, USE_REFRAC, curr_spk_evts, curr_gt_times,
        test_ind_input_duration, True, EVAL_LABEL, verbose=True, use_bef_tolerance=True
    )

    prediction_results.append(curr_eval_results)



--- HFO Evaluation - CH Index: 0 ---
lava_net: {'fc3': <lava.proc.dense.process.Dense object at 0x7f3274547940>, 'fc_in': <lava.proc.dense.process.Dense object at 0x7f3274546620>, 'fc_out': <lava.proc.dense.process.Dense object at 0x7f3274545cf0>, 'lif1': <lava.proc.lif.process.LIF object at 0x7f32745458a0>, 'lif2': <lava.proc.lif.process.LIF object at 0x7f32745453c0>, 'lif_out': <lava.proc.lif.process.LIF object at 0x7f3274544f10>}
startNodes: ['fc_in']
endNodes: ['lif_out']
Key: fc3 | Proc: <lava.proc.dense.process.Dense object at 0x7f3274547940>
Proc: Process_0 Port Name: s_in  Size: 24
Proc: Process_0 Port Name: a_out Size: 16
Key: fc_in | Proc: <lava.proc.dense.process.Dense object at 0x7f3274546620>
Proc: Process_1 Port Name: s_in  Size: 2
Proc: Process_1 Port Name: a_out Size: 24
Key: fc_out | Proc: <lava.proc.dense.process.Dense object at 0x7f3274545cf0>
Proc: Process_2 Port Name: s_in  Size: 16
Proc: Process_2 Port Name: a_out Size: 1
Key: lif1 | Proc: <lava.proc.lif.process

/home/monkin/Desktop/feup/thesis/thesis-lava/src/lava/magma/compiler/compiler_graphs.py:895: UserWarning: Cannot import module '<module 'snn' from '/home/monkin/Desktop/feup/thesis/thesis-lava/src/utils/snn.py'>' when searching ProcessModels for Process 'SpikeEventGen_v2'.
  warnings.warn(


Time step: 10000
Time step: 20000
Shape of LIF OUT Voltage Data: (24000, 1)
Found 7 spikes in the Output LIF Process
Spike time: 5420 (iter. 5420) at neuron: 0
Spike time: 8059 (iter. 8059) at neuron: 0
Spike time: 11895 (iter. 11895) at neuron: 0
Spike time: 15509 (iter. 15509) at neuron: 0
Spike time: 18561 (iter. 18561) at neuron: 0
Spike time: 20569 (iter. 20569) at neuron: 0
Spike time: 23067 (iter. 23067) at neuron: 0
gt_times Shape: (7,).
Preview: [ 5360.  8018. ... 20517. 23000.]
Found 7 real spikes in the Output LIF Process
[ 5420  8059 11895 15509 18561 20569 23067]
GT Time: 5360.0 | Spike Time: 5420
GT Time: 8018.0 | Spike Time: 8059
GT Time: 11859.0 | Spike Time: 11895
GT Time: 15439.0 | Spike Time: 15509
GT Time: 18551.0 | Spike Time: 18561
GT Time: 20517.0 | Spike Time: 20569
GT Time: 23000.0 | Spike Time: 23067
Confusion Matrix:
|TP: 7 | FP: 0|
|FN: 0  | TN: 0|
Recall (True Positive Rate): 100.00 %
Precision (TP / (TP + FP)): 100.00 %
F1 Score (Combines Precision & Recal

## Show the Prediction Results

In [24]:
print(f"prediction_results: {prediction_results}")

prediction_results: [{'label': 'CH Index: 0', 'TP': 7, 'FP': 0, 'FN': 0, 'TN': 0, 'recall': 1.0, 'precision': 1.0, 'f1_score': 1.0, 'total_predictions': 7}, {'label': 'CH Index: 1', 'TP': 4, 'FP': 1, 'FN': 0, 'TN': 0, 'recall': 1.0, 'precision': 0.8, 'f1_score': 0.888888888888889, 'total_predictions': 5}, {'label': 'CH Index: 2', 'TP': 4, 'FP': 0, 'FN': 0, 'TN': 0, 'recall': 1.0, 'precision': 1.0, 'f1_score': 1.0, 'total_predictions': 4}, {'label': 'CH Index: 3', 'TP': 5, 'FP': 0, 'FN': 0, 'TN': 0, 'recall': 1.0, 'precision': 1.0, 'f1_score': 1.0, 'total_predictions': 5}, {'label': 'CH Index: 4', 'TP': 3, 'FP': 1, 'FN': 3, 'TN': 0, 'recall': 0.5, 'precision': 0.75, 'f1_score': 0.6, 'total_predictions': 7}, {'label': 'CH Index: 5', 'TP': 4, 'FP': 1, 'FN': 1, 'TN': 0, 'recall': 0.8, 'precision': 0.8, 'f1_score': 0.8000000000000002, 'total_predictions': 6}, {'label': 'CH Index: 6', 'TP': 3, 'FP': 0, 'FN': 2, 'TN': 0, 'recall': 0.6, 'precision': 1.0, 'f1_score': 0.7499999999999999, 'total_

## Join the results from all channels

In [25]:
TP_ALL, FP_ALL, FN_ALL, TN_ALL = 0, 0, 0, 0
for ch_idx in range(len(prediction_results)):
    curr_results = prediction_results[ch_idx]
    TP_ALL += curr_results["TP"]
    FP_ALL += curr_results["FP"]
    FN_ALL += curr_results["FN"]
    TN_ALL += curr_results["TN"]

print(f"|TP: {TP_ALL} | FP: {FP_ALL}|\n|FN: {FN_ALL}  | TN: {TN_ALL}|")

|TP: 123 | FP: 10|
|FN: 26  | TN: 0|


In [26]:
# Calculate the performance metrics (Not including metrics that rely on TN)
recall = TP_ALL / (TP_ALL + FN_ALL) if (TP_ALL + FN_ALL) > 0 else 0
precision = TP_ALL / (TP_ALL + FP_ALL) if (TP_ALL + FP_ALL) > 0 else 0
f1_score = 2 * (precision * recall) / (precision +
                                        recall) if (precision + recall) > 0 else 0
total_predictions = TP_ALL + FP_ALL + FN_ALL
# Output the performance metrics
print(f"Recall (True Positive Rate): {recall*100:.2f} %")
print(f"Precision (TP / (TP + FP)): {precision*100:.2f} %")
print(f"F1 Score (Combines Precision & Recall): {f1_score*100:.2f} %")
print(f"Total Predictions: {total_predictions}")

Recall (True Positive Rate): 82.55 %
Precision (TP / (TP + FP)): 92.48 %
F1 Score (Combines Precision & Recall): 87.23 %
Total Predictions: 159


# Export the results of the Classification to a JSON file
Export the results of the classification to a JSON file. This file will include:
- Frequency Band used (`Ripple`, `Fast Ripple` or `Both`).
- Channels Used.
- `num_steps`
- `Confidence Window` used
- Classification Metrics (`True Positives`, `False Positives`, `False Negatives`, `Precision`, `Recall`, `F1 Score`)

In [27]:
from utils.hfo import band_to_gt_max_offset, band_to_file_name, MarkerType, BaselineAlgorithm

# Giving PRED_CAUSALITY_WINDOW ms for the network to update its inner state and spike
PRED_CAUSALITY_WINDOW = int(5)
MAX_DETECTION_OFFSET = int(band_to_gt_max_offset(
        chosen_band)) * 1.5 + PRED_CAUSALITY_WINDOW   # in timesteps (ms)

In [28]:
import json

# Export the results to a JSON file
OUTPUT_FOLDER = f"eval/{INPUT_TYPE}"
# create the output folder if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Create a dictionary with the results
json_results = {
    "freq_band": BAND_FILENAME,
    "channels": CH_SUFFIX,
    "max_detection_offset": MAX_DETECTION_OFFSET,
    "baseline_algorithm": chosen_baseline_alg_suffix,
    "metrics": {
        "true_positive": TP_ALL,
        "false_positive": FP_ALL,
        "false_negative": FN_ALL,
        "total_predictions": total_predictions,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
    }
}

EXPORT_JSON_FILE = True
if EXPORT_JSON_FILE:
    json_file_name = f"{OUTPUT_FOLDER}/fp_{BAND_FILENAME}_{chosen_baseline_alg_suffix}_{CH_SUFFIX}_results.json"
    with open(json_file_name, 'w') as f:
        json.dump(json_results, f)